# v1/v2 pokretanje i NCU analiza

Notebook za build, run i profilisanje sa `ncu` na istom scenariju.


## 0) GPU

In [ ]:
!nvidia-smi


## 1) Upload ZIP-a

In [ ]:
from google.colab import files
import os, zipfile, shutil

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('ZIP nije uploadan.')

zip_name = next(iter(uploaded.keys()))
extract_dir = '/content/work'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(extract_dir)

print('Upload zavrsen:', zip_name)
print('Raspakovano u:', extract_dir)


In [ ]:
import os

def is_valid_project_root(root):
    required = [
        os.path.join(root, 'v1', 'src', 'main.cu'),
        os.path.join(root, 'v2', 'src', 'main.cu'),
    ]
    return all(os.path.exists(p) for p in required)

def find_project_root(base='/content/work'):
    candidates = []
    for root, dirs, files in os.walk(base):
        if '__MACOSX' in root:
            continue
        if 'v1' in dirs and 'v2' in dirs and is_valid_project_root(root):
            candidates.append(root)

    if not candidates:
        return None

    candidates.sort(key=lambda p: (('__MACOSX' in p), len(p)))
    return candidates[0]

project_root = find_project_root()
if project_root is None:
    raise RuntimeError('Nisam nasao root sa v1/src/main.cu i v2/src/main.cu. Provjeri ZIP.')

os.chdir(project_root)
print('PROJECT_ROOT:', project_root)
print('Sadrzaj root foldera:', os.listdir(project_root))


## 2) Build i run v1


In [ ]:
!nvcc -O3 -std=c++17 v1/src/main.cu v1/src/kmeans_cpu.cpp -I v1/include -o v1/main
!./v1/main


## 3) Build i run v2 (isti scenario)


In [ ]:
!nvcc -O3 -std=c++17 v2/src/main.cu v2/src/kmeans_cpu.cpp -I v2/include -o v2/main
!./v2/main 8 256 131072 1024


## 4) NCU profilisanje (v1 i v2)

Scenario: `dim=8`, `clusters=256`, `points=131072`, `batchQ=1024`.


In [ ]:
SCENARIO_DIM = 8
SCENARIO_CLUSTERS = 256
SCENARIO_POINTS = 131072
SCENARIO_BATCHQ = 1024
print('Scenario parametri:', SCENARIO_DIM, SCENARIO_CLUSTERS, SCENARIO_POINTS, SCENARIO_BATCHQ)


In [ ]:
!which ncu || true
!ncu --version || true


In [ ]:
%%bash
set -euo pipefail

if command -v ncu >/dev/null 2>&1; then
  NCU_BIN="$(command -v ncu)"
elif [ -x /usr/local/cuda/bin/ncu ]; then
  NCU_BIN=/usr/local/cuda/bin/ncu
else
  echo "ncu nije dostupan u ovom runtime-u."
  exit 1
fi

echo "NCU bin: $NCU_BIN"
"$NCU_BIN" --target-processes all --force-overwrite --set full   --section SpeedOfLight   --section Occupancy   --section MemoryWorkloadAnalysis   --section SchedulerStats   --section WarpStateStats   --section LaunchStats   --kernel-name-base demangled   --csv -o v1_default_ncu   ./v1/main > v1_default_ncu.csv

echo 'Sacuvan: v1_default_ncu.csv'
head -n 40 v1_default_ncu.csv || true


In [ ]:
%%bash
set -euo pipefail

if command -v ncu >/dev/null 2>&1; then
  NCU_BIN="$(command -v ncu)"
elif [ -x /usr/local/cuda/bin/ncu ]; then
  NCU_BIN=/usr/local/cuda/bin/ncu
else
  echo "ncu nije dostupan u ovom runtime-u."
  exit 1
fi

echo "NCU bin: $NCU_BIN"
"$NCU_BIN" --target-processes all --force-overwrite --set full   --section SpeedOfLight   --section Occupancy   --section MemoryWorkloadAnalysis   --section SchedulerStats   --section WarpStateStats   --section LaunchStats   --kernel-name-base demangled   --csv -o v2_default_ncu   ./v2/main 8 256 131072 1024 > v2_default_ncu.csv

echo 'Sacuvan: v2_default_ncu.csv'
head -n 40 v2_default_ncu.csv || true


## 5) Ispis metrika iz `.ncu-rep`

CSV export + pregled ključnih metrika po fazama.


In [ ]:
%%bash
set -euo pipefail

if command -v ncu >/dev/null 2>&1; then
  NCU_BIN="$(command -v ncu)"
elif [ -x /usr/local/cuda/bin/ncu ]; then
  NCU_BIN=/usr/local/cuda/bin/ncu
else
  echo "ncu nije dostupan."
  exit 1
fi

"$NCU_BIN" --import v1_default_ncu.ncu-rep --csv --page raw > v1_raw.csv
"$NCU_BIN" --import v2_default_ncu.ncu-rep --csv --page raw > v2_raw.csv

echo "Raw CSV export:"
wc -l v1_raw.csv v2_raw.csv


In [ ]:
import io
import pandas as pd

PHASE_ORDER = ["AB", "C", "D", "EF"]

def load_ncu_wide_csv(path):
    txt = open(path, "r", encoding="utf-8", errors="ignore").read()
    lines = []
    for ln in txt.splitlines():
        if ln.startswith("==PROF=="):
            continue
        if not ln.strip():
            continue
        lines.append(ln)
    return pd.read_csv(io.StringIO("\n".join(lines)), on_bad_lines="skip")

def pick_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def series_num(df, col):
    if col is None or col not in df.columns:
        return pd.Series([pd.NA] * len(df), index=df.index, dtype="float64")
    return pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")

def clean_kernel_name(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    return s

def phase_from_kernel(name):
    s = str(name).lower()
    if "dist2qc_and_seed" in s:
        return "AB"
    if "dk_from_seed" in s:
        return "C"
    if "build_active_from_dist2" in s:
        return "D"
    if "topk_from_active" in s:
        return "EF"
    return None

def clamp_metrics(df):
    out = df.copy()
    for c in ["sm_pct", "dram_pct", "l2_hit_pct"]:
        if c in out.columns:
            out[c] = out[c].clip(lower=0, upper=100)
    return out

def aggregate_by_kernel(df):
    kcol = pick_existing(df, ["Kernel Name", "launch__kernel_name"])
    if kcol is None:
        raise RuntimeError(f"Nema kolone sa imenom kernela. Kolone: {list(df.columns)[:20]}")

    cols = {
        "time_raw": pick_existing(df, ["gpu__time_duration.sum", "gpu__time_duration.avg"]),
        "sm_pct": pick_existing(df, ["sm__throughput.avg.pct_of_peak_sustained_elapsed", "sm__throughput.avg.pct_of_peak_sustained_active"]),
        "dram_pct": pick_existing(df, ["gpu__dram_throughput.avg.pct_of_peak_sustained_elapsed", "gpu__dram_throughput.sum.pct_of_peak_sustained_elapsed"]),
        "l2_hit_pct": pick_existing(df, ["lts__t_sector_hit_rate.pct"]),
        "regs": pick_existing(df, ["launch__registers_per_thread", "launch__registers_per_thread_allocated"]),
        "stall_long_sb": pick_existing(df, ["smsp__average_warps_issue_stalled_long_scoreboard_per_issue_active.ratio"]),
        "stall_short_sb": pick_existing(df, ["smsp__average_warps_issue_stalled_short_scoreboard_per_issue_active.ratio"]),
        "stall_barrier": pick_existing(df, ["smsp__average_warps_issue_stalled_barrier_per_issue_active.ratio"]),
        "stall_wait": pick_existing(df, ["smsp__average_warps_issue_stalled_wait_per_issue_active.ratio"]),
        "stall_not_selected": pick_existing(df, ["smsp__average_warps_issue_stalled_not_selected_per_issue_active.ratio"]),
    }

    work = pd.DataFrame({"kernel": df[kcol].map(clean_kernel_name)})
    for out, src in cols.items():
        work[out] = series_num(df, src)

    work = work[work["kernel"].notna()].copy()
    if work.empty:
        raise RuntimeError("Nema validnih kernel redova poslije filtriranja.")

    grouped = work.groupby("kernel", dropna=False)
    agg = pd.DataFrame(index=grouped.size().index)
    agg["time_raw"] = grouped["time_raw"].sum(min_count=1)
    for c in [
        "sm_pct", "dram_pct", "l2_hit_pct", "regs", "stall_long_sb", "stall_short_sb",
        "stall_barrier", "stall_wait", "stall_not_selected"
    ]:
        agg[c] = grouped[c].mean()

    agg = clamp_metrics(agg)
    agg = agg.sort_values("time_raw", ascending=False)

    total = agg["time_raw"].sum()
    agg["time_pct"] = (agg["time_raw"] / total * 100.0) if pd.notna(total) and total > 0 else pd.NA

    max_time = agg["time_raw"].max()
    if pd.isna(max_time):
        agg["time_est_ms"] = pd.NA
    elif max_time > 1e6:
        agg["time_est_ms"] = agg["time_raw"] / 1e6
    elif max_time > 1e3:
        agg["time_est_ms"] = agg["time_raw"] / 1e3
    else:
        agg["time_est_ms"] = agg["time_raw"]

    return agg

def aggregate_by_phase(kernel_agg):
    x = kernel_agg.copy()
    x["phase"] = [phase_from_kernel(k) for k in x.index]
    x = x[x["phase"].notna()].copy()

    def wavg(sub, col):
        w = sub["time_raw"]
        v = sub[col]
        m = w.notna() & v.notna()
        if not m.any():
            return pd.NA
        ww = w[m]
        vv = v[m]
        sw = ww.sum()
        if sw == 0 or pd.isna(sw):
            return vv.mean()
        return (vv * ww).sum() / sw

    rows = []
    for phase, sub in x.groupby("phase", dropna=False):
        row = {
            "phase": phase,
            "time_raw": sub["time_raw"].sum(min_count=1),
            "time_est_ms": sub["time_est_ms"].sum(min_count=1),
            "time_pct": sub["time_pct"].sum(min_count=1),
        }
        for col in [
            "sm_pct", "dram_pct", "l2_hit_pct", "regs", "stall_long_sb", "stall_short_sb",
            "stall_barrier", "stall_wait", "stall_not_selected"
        ]:
            row[col] = wavg(sub, col)
        rows.append(row)

    out = pd.DataFrame(rows).set_index("phase")
    out = clamp_metrics(out)
    out["_ord"] = out.index.map(lambda p: PHASE_ORDER.index(p) if p in PHASE_ORDER else 999)
    out = out.sort_values("_ord").drop(columns=["_ord"])
    return out

def add_quality_flags(df):
    out = df.copy()
    flags = []
    for _, r in out.iterrows():
        f = []
        if pd.notna(r.get("dram_pct")) and r["dram_pct"] > 50:
            f.append("high_dram")
        if pd.notna(r.get("l2_hit_pct")) and r["l2_hit_pct"] < 85:
            f.append("low_l2")
        if pd.notna(r.get("stall_long_sb")) and r["stall_long_sb"] > 10:
            f.append("long_sb")
        if pd.notna(r.get("regs")) and r["regs"] > 64:
            f.append("high_regs")
        flags.append(",".join(f) if f else "ok")
    out["flags"] = flags
    return out

def print_summary(label, kernel_agg, phase_agg):
    print(f"\n--- {label} ---")

    top = kernel_agg.head(6)[[
        "time_est_ms", "time_pct", "sm_pct", "dram_pct", "l2_hit_pct", "regs",
        "stall_long_sb", "stall_short_sb", "stall_barrier", "stall_wait", "stall_not_selected"
    ]]
    top = add_quality_flags(top)
    print("Top kerneli:")
    print(top.to_string(float_format=lambda v: f"{v:,.3f}"))

    cols = [
        "time_est_ms", "time_pct", "sm_pct", "dram_pct", "l2_hit_pct", "regs",
        "stall_long_sb", "stall_short_sb", "stall_barrier", "stall_wait", "stall_not_selected"
    ]
    p = add_quality_flags(phase_agg[cols])
    print("\nFaze (AB/C/D/EF):")
    print(p.to_string(float_format=lambda v: f"{v:,.3f}"))

def compare_phase(v1_phase, v2_phase):
    keep = [
        "time_pct", "sm_pct", "dram_pct", "l2_hit_pct", "regs",
        "stall_long_sb", "stall_short_sb", "stall_barrier", "stall_wait", "stall_not_selected"
    ]
    cmp = v1_phase[keep].add_suffix("_v1").join(v2_phase[keep].add_suffix("_v2"), how="outer")
    for c in keep:
        cmp[f"delta_{c}"] = cmp[f"{c}_v2"] - cmp[f"{c}_v1"]

    print("\n--- Razlika v2 - v1 (po fazama) ---")
    delta_cols = [f"delta_{c}" for c in keep]
    print(cmp[delta_cols].to_string(float_format=lambda v: f"{v:,.3f}"))
    return cmp

v1_kernel = aggregate_by_kernel(load_ncu_wide_csv("v1_raw.csv"))
v2_kernel = aggregate_by_kernel(load_ncu_wide_csv("v2_raw.csv"))

v1_phase = aggregate_by_phase(v1_kernel)
v2_phase = aggregate_by_phase(v2_kernel)

print_summary("v1", v1_kernel, v1_phase)
print_summary("v2", v2_kernel, v2_phase)
_ = compare_phase(v1_phase, v2_phase)



## 6) Download NCU CSV fajlova


In [ ]:
from google.colab import files
import os

for f in [
    'v1_default_ncu.csv',
    'v2_default_ncu.csv',
]:
    if os.path.exists(f):
        print('Preuzimanje:', f)
        files.download(f)
    else:
        print('Nije pronadjen:', f)


## 7)  v2 test scenariji


In [ ]:
!chmod +x v2/tests/run_scenarios.sh
!v2/tests/run_scenarios.sh ./v2/main


## 8) Grafovi scenarija


In [ ]:
import pandas as pd

data = [
    {"scenario":"synth_tiny",          "dim":8,  "clusters":64,   "points":32768,  "per_cluster":512,  "batchQ":512,  "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":72.540,   "gpu_ms":2.929,  "active_avg":1.50, "active_max":2},
    {"scenario":"synth_small_dim",     "dim":4,  "clusters":256,  "points":131072, "per_cluster":512,  "batchQ":1024, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":786.407,  "gpu_ms":3.731,  "active_avg":2.00, "active_max":2},
    {"scenario":"synth_exact_16",      "dim":16, "clusters":256,  "points":131072, "per_cluster":512,  "batchQ":1024, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":2281.852, "gpu_ms":4.095,  "active_avg":2.50, "active_max":3},
    {"scenario":"synth_large_dim",     "dim":12, "clusters":512,  "points":262144, "per_cluster":512,  "batchQ":1024, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":6141.170, "gpu_ms":5.184,  "active_avg":1.00, "active_max":1},
    {"scenario":"synth_huge",          "dim":8,  "clusters":1024, "points":524288, "per_cluster":512,  "batchQ":2048, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":19179.546,"gpu_ms":8.668,  "active_avg":1.00, "active_max":1},
    {"scenario":"low_occupancy",       "dim":8,  "clusters":512,  "points":65536,  "per_cluster":128,  "batchQ":2048, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":1070.919, "gpu_ms":5.628,  "active_avg":2.00, "active_max":2},
    {"scenario":"high_dim_stress",     "dim":16, "clusters":512,  "points":262144, "per_cluster":512,  "batchQ":1024, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":8270.136, "gpu_ms":5.141,  "active_avg":1.00, "active_max":1},
    {"scenario":"cluster_imbalance",   "dim":8,  "clusters":128,  "points":262144, "per_cluster":2048, "batchQ":1024, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":1096.295, "gpu_ms":7.494,  "active_avg":1.50, "active_max":2},
    {"scenario":"degenerate_clusters", "dim":8,  "clusters":512,  "points":131072, "per_cluster":256,  "batchQ":1024, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":2129.259, "gpu_ms":5.083,  "active_avg":1.00, "active_max":1},
    {"scenario":"worst_pruning",       "dim":8,  "clusters":256,  "points":131072, "per_cluster":512,  "batchQ":8192, "k":20, "sigma":0.08, "min_center_dist":4.0, "cpu_ms":1065.336, "gpu_ms":15.979, "active_avg":2.00, "active_max":2},
]

df = pd.DataFrame(data)
df["gpu_us_per_query"] = df["gpu_ms"] * 1000 / df["batchQ"]
df["qps"] = df["batchQ"] / (df["gpu_ms"] / 1000.0)
df["cpu_gpu_speedup"] = df["cpu_ms"] / df["gpu_ms"]

df



## 9) Speedup graf

Ubrzanje po scenariju (`cpu_ms / gpu_ms`) sa dodatnim prikazom GPU vremena.


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

if 'df' not in globals():
    raise RuntimeError('Prvo pokreni sekciju 8 (df tabela).')

os.makedirs('figures', exist_ok=True)

short_map = {
    'synth_tiny': 'Tiny',
    'synth_small_dim': 'Small-D',
    'synth_exact_16': 'D=16',
    'synth_large_dim': 'Large-D',
    'synth_huge': 'Huge',
    'low_occupancy': 'Low-Occ',
    'high_dim_stress': 'High-D',
    'cluster_imbalance': 'Imbalance',
    'degenerate_clusters': 'Degenerate',
    'worst_pruning': 'Worst-Prune',
}

plot_df = df.copy().sort_values('cpu_gpu_speedup', ascending=True)
plot_df['label'] = plot_df['scenario'].map(short_map).fillna(plot_df['scenario'])

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': 300,
})

# Color by dim for quick grouping
unique_dims = sorted(plot_df['dim'].unique())
cmap = plt.cm.tab10
color_map = {d: cmap(i % 10) for i, d in enumerate(unique_dims)}
colors = [color_map[d] for d in plot_df['dim']]

fig, ax = plt.subplots(figsize=(8.2, 5.0), constrained_layout=True)

y = np.arange(len(plot_df))
vals = plot_df['cpu_gpu_speedup'].values
bars = ax.barh(y, vals, color=colors, edgecolor='black', linewidth=0.7, alpha=0.95)

ax.set_yticks(y)
ax.set_yticklabels(plot_df['label'])
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.9)

xmax = vals.max()
for bar, spd, gms in zip(bars, plot_df['cpu_gpu_speedup'], plot_df['gpu_ms']):
    x = bar.get_width()
    y0 = bar.get_y() + bar.get_height() / 2
    ax.text(x + 0.015 * xmax, y0, f'{spd:.1f}x', va='center', ha='left', fontsize=9)
    ax.text(x + 0.12 * xmax, y0, f'GPU {gms:.2f} ms', va='center', ha='left', fontsize=8, color='0.35')

ax.set_title('CPU/GPU ubrzanje po scenariju')
ax.set_xlabel('Ubrzanje (CPU vrijeme / GPU vrijeme)')
ax.set_ylabel('Scenarij')
ax.xaxis.set_major_locator(MaxNLocator(nbins=9))
ax.grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, xmax * 1.45)

handles = [Patch(facecolor=color_map[d], edgecolor='black', label=f'dim={d}') for d in unique_dims]
ax.legend(handles=handles, title='Dimenzija', loc='lower right', frameon=True, ncol=2)

fig.savefig('figures/fig05_speedup_by_scenario.pdf', bbox_inches='tight')
fig.savefig('figures/fig05_speedup_by_scenario.png', bbox_inches='tight')
plt.show()

